# Extended Data Cleaning and Preprocessing

## Mental Wellness & Nutrition Project

### Objective
Extend the existing cleaned dataset by processing the remaining
unstandardized variables while preserving previously completed cleaning.

### Dataset
Input: cleaned_file.csv.xls

Output: extended_cleaned_data.csv

In [1]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
from pathlib import Path

In [2]:
# Load the existing cleaned dataset
DATA_PATH = Path("cleaned_file.csv.xls")

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

display(df.head())

Dataset loaded successfully!
Shape: (100, 25)


,respondent_id,submitted_at,gender,academic_year,living_arrangement,academic_screen_raw,entertainment_screen_raw,stress_anxious,racing_thoughts,concentration,...,exercise_freq,sitting_raw,evening_energy,water_raw,meal_regularity,diet_raw,digestive_discomfort,sleep_hours,screen_hours,water_litres
0,1,2026-08-14 19:04:33.748,Male,2nd year,Hostel,3-4 hours,1-2 hours,2,4,4,...,3-4 days,1-2 hours,3,2-3 litres,Skip 1 meal daily,"High-Protein items (Eggs, Paneer, Meat, Pulses...",2,5.0,1.5,2.5
1,2,2026-08-14 19:18:55.166,Male,2nd year,Hostel,6 hr,3 hr,1,5,1,...,3-4 days,2 hr,3,1 liter,3 regular meals,"Fresh Fruits / Vegetables, Home-cooked / Mess-...",2,7.0,3.0,1.0
2,3,2026-08-14 19:20:19.130,Female,2nd year,Flat/PG,10,4,5,5,5,...,0 days,14,1,No idea,Irregular/Random eating times,Sugary Drinks / Energy Drinks / Excessive Caff...,2,5.0,4.0,3.0
3,4,2026-08-14 19:21:30.282,Male,2nd year,Hostel,4,6,3,4,4,...,0 days,30 min,2,2,Skip 1 meal daily,Home-cooked / Mess-cooked balanced meals,2,5.0,6.0,2.0
4,5,2026-08-14 19:21:57.839,Male,3rd year,Flat/PG,3-4hrs,2-3hrs,5,3,5,...,5+ days,Mainly 4-5hrs,4,3ltr,Irregular/Random eating times,"Fast Food / Fried Snacks / Junk Food, Home-coo...",4,8.5,2.5,3.0


In [3]:
# Preserve the original loaded dataset
df_extended = df.copy(deep=True)

print("Original dataset:", df.shape)
print("Working dataset:", df_extended.shape)

Original dataset: (100, 25)
Working dataset: (100, 25)


## Inspection of Remaining Numerical Variables

The following variables have not yet been standardized:

1. Academic screen time
2. Daily steps / walking duration
3. Consecutive sitting duration

We will examine their raw response formats before applying
appropriate cleaning rules.

In [4]:
remaining_numeric = [
    "academic_screen_raw",
    "steps_raw",
    "sitting_raw"
]

for col in remaining_numeric:
    print(f"\n{'=' * 50}")
    print(f"Column: {col}")
    print(f"{'=' * 50}")

    print("\nUnique responses:")
    print(df_extended[col].value_counts(dropna=False).to_string())

    print("\nTotal unique values:",
          df_extended[col].nunique(dropna=False))


Column: academic_screen_raw

Unique responses:
academic_screen_raw
4                    9
2                    8
6                    8
3                    7
5                    4
1                    4
4 hours              3
6 hr                 2
10                   2
7 hours              2
9                    2
4-5                  2
7                    2
3-4                  2
2hr                  2
6-7 hours            2
8 hours              2
4hrs                 2
3-4 hours            1
3-4hrs               1
2.5hrs               1
1 hour.              1
7 hours 2 mins       1
4-5 hours            1
1/8 hour             1
5 hours              1
4 to 5               1
4hours               1
6hours               1
15                   1
8                    1
3 hours approx       1
4 hour               1
2hours               1
8hrs                 1
7-8                  1
3 to 4 hours         1
3-4 hrs              1
7-8 hrs              1
5 hrs                1
8-9         

In [5]:

def clean_academic_screen(x):
    if pd.isna(x):
        return np.nan

    s = str(x).lower().strip()

    # Unknown or ambiguous responses
    if s in [
        "don't know",
        "dont know",
        "not specific time"
    ]:
        return np.nan

    # Slash notation is ambiguous
    if "/" in s:
        return np.nan

    # Hours and minutes together
    match = re.search(
        r"(\d+(?:\.\d+)?)\s*(?:hours?|hrs?|hr)\s*"
        r"(\d+(?:\.\d+)?)\s*(?:minutes?|mins?|min)",
        s
    )

    if match:
        return float(match.group(1)) + float(match.group(2)) / 60

    # Ranges: 3-4, 3 to 4, etc.
    match = re.search(
        r"(\d+(?:\.\d+)?)\s*(?:-|to)\s*(\d+(?:\.\d+)?)",
        s
    )

    if match:
        return (
            float(match.group(1)) +
            float(match.group(2))
        ) / 2

    # Single numerical value
    match = re.search(r"\d+(?:\.\d+)?", s)

    if match:
        return float(match.group())

    return np.nan


df_extended["academic_screen_hours"] = (
    df_extended["academic_screen_raw"]
    .apply(clean_academic_screen)
)

print("Academic Screen Time Cleaning Results")
print("Valid:",
      df_extended["academic_screen_hours"].notna().sum())
print("Missing:",
      df_extended["academic_screen_hours"].isna().sum())

display(
    df_extended[
        ["academic_screen_raw", "academic_screen_hours"]
    ].head(20)
)


Academic Screen Time Cleaning Results
Valid: 97
Missing: 3


,academic_screen_raw,academic_screen_hours
0,3-4 hours,3.500000
1,6 hr,6.000000
2,10,10.000000
3,4,4.000000
4,3-4hrs,3.500000
5,7 hours,7.000000
6,2.5hrs,2.500000
7,1 hour.,1.000000
8,2,2.000000
9,2,2.000000


In [6]:
print("Missing academic screen values:",
      df_extended["academic_screen_hours"].isna().sum())

print("Academic screen time summary:")
print(df_extended["academic_screen_hours"].describe())

Missing academic screen values: 3
Academic screen time summary:
count    97.000000
mean      4.624055
std       2.405885
min       1.000000
25%       3.000000
50%       4.000000
75%       6.000000
max      15.000000
Name: academic_screen_hours, dtype: float64


In [7]:
def clean_sitting(x):
    if pd.isna(x):
        return np.nan

    s = str(x).lower().strip()

    unknown = [
        "dont know", "don't know",
        "dont know excatly", "no idea", "jssj"
    ]

    if s in unknown:
        return np.nan

    if s == "half an hour":
        return 0.5

    if s == "one or half hr":
        return np.nan  # Ambiguous

    # Long paragraph: extract stated consecutive sitting duration
    if len(s) > 100:
        match = re.search(r"around\s+(\d+)\s+to\s+(\d+)\s+hours", s)
        if match:
            return (float(match.group(1)) +
                    float(match.group(2))) / 2
        return np.nan

    # Handle 1:30-2 hours
    s = re.sub(r"\b1:30\b", "1.5", s)

    numbers = re.findall(r"\d+(?:\.\d+)?", s)

    if not numbers:
        return np.nan

    value = (
        (float(numbers[0]) + float(numbers[1])) / 2
        if len(numbers) >= 2
        else float(numbers[0])
    )

    if "min" in s:
        value /= 60

    return value


df_extended["sitting_hours"] = (
    df_extended["sitting_raw"].apply(clean_sitting)
)

print("Sitting duration:")
print("Valid:", df_extended["sitting_hours"].notna().sum())
print("Missing:", df_extended["sitting_hours"].isna().sum())

Sitting duration:
Valid: 94
Missing: 6


In [8]:
def clean_activity(x):
    if pd.isna(x):
        return (np.nan, np.nan, np.nan)

    s = str(x).lower().strip().replace(",", "")

    unknown = [
        "djdj", "idk", "na",
        "dont know", "don't know",
        "doesn't count", "no specifically know"
    ]

    if s in unknown or "not sure" in s:
        return (np.nan, np.nan, np.nan)

    # Standardize number suffixes
    s = re.sub(r"(\d+)\s*k\b", r"\1 thousand", s)

    numbers = re.findall(r"\d+(?:\.\d+)?", s)

    if not numbers:
        return (np.nan, np.nan, np.nan)

    values = [float(v) for v in numbers]

    value = (
        sum(values[:2]) / 2
        if len(values) >= 2
        else values[0]
    )

    if "thousand" in s:
        value *= 1000

    # Distance
    if "km" in s:
        return (np.nan, np.nan, value)

    # Duration
    if any(unit in s for unit in ["min", "hour", "hr"]):
        if "hour" in s or "hr" in s:
            value *= 60
        return (np.nan, value, np.nan)

    # Explicit steps
    if any(unit in s for unit in ["step", "count"]):
        return (value, np.nan, np.nan)

    # Unitless values are ambiguous
    return (np.nan, np.nan, np.nan)


activity = df_extended["steps_raw"].apply(clean_activity)

df_extended[
    ["daily_steps", "walking_minutes", "walking_km"]
] = pd.DataFrame(
    activity.tolist(),
    index=df_extended.index
)

print("Daily steps:", df_extended["daily_steps"].notna().sum())
print("Walking minutes:", df_extended["walking_minutes"].notna().sum())
print("Walking distance:", df_extended["walking_km"].notna().sum())

Daily steps: 22
Walking minutes: 21
Walking distance: 1


In [9]:
df_extended["main_factors_clean"] = (
    df_extended["main_factors"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
)

print("Non-empty text responses:",
      df_extended["main_factors_clean"].ne("").sum())

Non-empty text responses: 99


In [10]:

# Identify responses not assigned to any activity measurement

unresolved = df_extended[
    df_extended[
        ["daily_steps", "walking_minutes", "walking_km"]
    ].isna().all(axis=1)
]

print("Unresolved activity responses:", len(unresolved))

print("\nUnresolved response frequencies:")
print(
    unresolved["steps_raw"]
    .value_counts(dropna=False)
    .to_string()
)


Unresolved activity responses: 56

Unresolved response frequencies:
steps_raw
10000                                      6
30                                         4
60                                         4
7000                                       3
8000                                       3
45                                         2
120                                        2
10k+                                       2
8-10 thousand                              1
8k                                         1
Not sure but maybe around 4-5 kms a day    1
50-60                                      1
2k                                         1
17000                                      1
7-9k                                       1
Djdj                                       1
5-6K                                       1
900-1000                                   1
1000                                       1
10000 to 15000                             1
20 maybe              

In [11]:

def infer_activity(x):
    if pd.isna(x):
        return (np.nan, "unknown")

    s = str(x).lower().strip().replace(",", "")

    unknown = [
        "djdj", "idk", "na",
        "dont know", "don't know",
        "doesn't count", "no specifically know"
    ]

    if s in unknown:
        return (np.nan, "unknown")

    if any(unit in s for unit in [
        "step", "count", "min", "hour", "hr", "km"
    ]):
        return (np.nan, "already_classified")

    s = re.sub(r"(\d+)\s*k\b", r"\1 thousand", s)

    nums = [
        float(n)
        for n in re.findall(r"\d+(?:\.\d+)?", s)
    ]

    if not nums:
        return (np.nan, "unknown")

    value = (
        sum(nums[:2]) / 2
        if len(nums) >= 2
        else nums[0]
    )

    if "thousand" in s:
        value *= 1000

    if value >= 1000:
        return (value, "probable_steps")

    if 0 < value <= 180:
        return (value, "probable_minutes")

    return (value, "ambiguous")


inferred = df_extended["steps_raw"].apply(infer_activity)

df_extended[
    ["activity_inferred_value", "activity_inferred_unit"]
] = pd.DataFrame(
    inferred.tolist(),
    index=df_extended.index
)

print(df_extended["activity_inferred_unit"].value_counts())


activity_inferred_unit
already_classified    45
probable_steps        30
probable_minutes      16
unknown                7
ambiguous              2
Name: count, dtype: int64


In [12]:

def activity_status(row):
    if pd.notna(row["daily_steps"]):
        return "confirmed_steps"

    if pd.notna(row["walking_minutes"]):
        return "confirmed_minutes"

    if pd.notna(row["walking_km"]):
        return "confirmed_distance"

    if row["activity_inferred_unit"] == "probable_steps":
        return "probable_steps"

    if row["activity_inferred_unit"] == "probable_minutes":
        return "probable_minutes"

    if row["activity_inferred_unit"] == "ambiguous":
        return "ambiguous"

    return "unknown"


df_extended["activity_status"] = df_extended.apply(
    activity_status, axis=1
)

print(df_extended["activity_status"].value_counts())


activity_status
probable_steps        30
confirmed_steps       22
confirmed_minutes     21
probable_minutes      16
unknown                8
ambiguous              2
confirmed_distance     1
Name: count, dtype: int64


In [13]:
# Map full category prefixes to clean feature names
diet_mapping = {
    "Carbohydrate-Rich Foods": "diet_carbohydrates",
    "Dietary & Health Supplements": "diet_supplements",
    "High-Protein items": "diet_high_protein",
    "Home-cooked / Mess-cooked balanced meals": "diet_balanced_meals",
    "Fast Food / Fried Snacks / Junk Food": "diet_fast_food",
    "Fresh Fruits / Vegetables": "diet_fruits_vegetables",
    "Sugary Drinks / Energy Drinks / Excessive Caffeine": "diet_sugary_caffeine"
}

for category, feature in diet_mapping.items():
    df_extended[feature] = (
        df_extended["diet_raw"]
        .fillna("")
        .astype(str)
        .str.contains(
            re.escape(category),
            case=False,
            regex=True
        )
        .astype(int)
    )

print("dietary features:\n")

for feature in diet_mapping.values():
    print(
        f"{feature}:",
        int(df_extended[feature].sum())
    )


dietary features:

diet_carbohydrates: 60
diet_supplements: 18
diet_high_protein: 32
diet_balanced_meals: 73
diet_fast_food: 44
diet_fruits_vegetables: 33
diet_sugary_caffeine: 29


In [14]:

# ============================================================
# FINAL EXTENDED DATA CLEANING - CORRECTIONS & AUDIT
# ============================================================

# 1. Correct known step-count parsing error
typo_mask = (
    df_extended["steps_raw"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("10thoudand steps")
)

df_extended.loc[typo_mask, "daily_steps"] = 10000
df_extended.loc[typo_mask, "activity_status"] = "confirmed_steps"

print("Corrected step-count records:", typo_mask.sum())


# 2. Flag lower-bound activity responses
df_extended["activity_lower_bound"] = (
    df_extended["steps_raw"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("10k+")
)

print("Lower-bound responses:",
      df_extended["activity_lower_bound"].sum())


# 3. Dataset structure and integrity
print("\n========== DATASET STRUCTURE ==========")

print("Dataset shape:", df_extended.shape)

print(
    "Duplicate respondent IDs:",
    df_extended["respondent_id"].duplicated().sum()
)

print(
    "Missing respondent IDs:",
    df_extended["respondent_id"].isna().sum()
)


# 4. Numerical validity checks
print("\n========== NUMERICAL VALIDITY ==========")

for col in ["academic_screen_hours", "sitting_hours"]:

    values = pd.to_numeric(
        df_extended[col], errors="coerce"
    )

    invalid = (
        values.notna()
        & (
            ~np.isfinite(values)
            | (values < 0)
            | (values > 24)
        )
    )

    print(f"\n{col}")
    print("Valid:", int((values.notna() & ~invalid).sum()))
    print("Missing:", int(values.isna().sum()))
    print("Invalid:", int(invalid.sum()))


# 5. Activity measurements
print("\n========== ACTIVITY MEASUREMENTS ==========")

for col in ["daily_steps", "walking_minutes", "walking_km"]:
    print(f"\n{col}")
    print(df_extended[col].describe())


# 6. Flag activity responses needing review
review_pattern = (
    r"\bk\+|\d+k\b|thousand|thoudand|"
    r"\bkm\b|kms|maybe|around|to"
)

review = df_extended[
    df_extended["steps_raw"]
    .astype(str)
    .str.contains(
        review_pattern,
        case=False,
        regex=True
    )
]

print("\nActivity responses flagged for review:", len(review))

display(
    review[
        [
            "steps_raw",
            "daily_steps",
            "walking_minutes",
            "walking_km",
            "activity_status",
            "activity_lower_bound"
        ]
    ]
)


# 7. Dietary encoding validation
print("\n========== DIETARY ENCODING ==========")

diet_cols = [
    col for col in df_extended.columns
    if col.startswith("diet_") and col != "diet_raw"
]

print("Dietary binary features:", len(diet_cols))

for col in diet_cols:
    assert df_extended[col].isin([0, 1]).all(), (
        f"Invalid binary values in {col}"
    )

print("All dietary features contain only 0 and 1.")


# 8. Verify original dataset columns are preserved
print("\n========== ORIGINAL DATA PRESERVATION ==========")

original_cols = [
    col for col in df.columns
    if col in df_extended.columns
]

unchanged = df[original_cols].equals(
    df_extended[original_cols]
)

print("Original columns unchanged:", unchanged)


# 9. Generate final audit summary
audit_summary = pd.DataFrame({
    "Check": [
        "Dataset rows",
        "Dataset columns",
        "Duplicate respondent IDs",
        "Missing respondent IDs",
        "Missing academic screen values",
        "Missing sitting duration values",
        "Dietary binary features",
        "Corrected step-count records",
        "Lower-bound activity responses",
        "Activity responses flagged for review",
        "Original columns preserved"
    ],
    "Result": [
        len(df_extended),
        len(df_extended.columns),
        df_extended["respondent_id"].duplicated().sum(),
        df_extended["respondent_id"].isna().sum(),
        df_extended["academic_screen_hours"].isna().sum(),
        df_extended["sitting_hours"].isna().sum(),
        len(diet_cols),
        int(typo_mask.sum()),
        int(df_extended["activity_lower_bound"].sum()),
        len(review),
        unchanged
    ]
})

print("\n========== FINAL AUDIT SUMMARY ==========")
display(audit_summary)


# 10. Save audit report and corrected dataset
audit_summary.to_excel(
    "extended_cleaning_audit.xlsx",
    index=False
)

df_extended.to_excel(
    "extended_cleaned_data.xlsx",
    index=False
)

print("\nFiles saved successfully!")
print("Final dataset shape:", df_extended.shape)

# Verify saved dataset
verified = pd.read_csv("extended_cleaned_data.csv")

print("Reloaded shape:", verified.shape)
print(
    "Shapes match:",
    verified.shape == df_extended.shape
)


Corrected step-count records: 1
Lower-bound responses: 2

========== DATASET STRUCTURE ==========
Dataset shape: (100, 42)
Duplicate respondent IDs: 0
Missing respondent IDs: 0

========== NUMERICAL VALIDITY ==========

academic_screen_hours
Valid: 97
Missing: 3
Invalid: 0

sitting_hours
Valid: 94
Missing: 6
Invalid: 0

========== ACTIVITY MEASUREMENTS ==========

daily_steps
count       22.000000
mean      7859.090909
std       3876.014041
min        200.000000
25%       5625.000000
50%       8000.000000
75%      10000.000000
max      15000.000000
Name: daily_steps, dtype: float64

walking_minutes
count     21.000000
mean      84.190476
std      106.243879
min        5.000000
25%       30.000000
50%       60.000000
75%       90.000000
max      500.000000
Name: walking_minutes, dtype: float64

walking_km
count    1.0
mean     5.0
std      NaN
min      5.0
25%      5.0
50%      5.0
75%      5.0
max      5.0
Name: walking_km, dtype: float64

Activity responses flagged for review: 16


,steps_raw,daily_steps,walking_minutes,walking_km,activity_status,activity_lower_bound
0,8-10 thousand,NaN,NaN,NaN,probable_steps,False
4,Average daily steps 15k,15000.0,NaN,NaN,confirmed_steps,False
6,8k,NaN,NaN,NaN,probable_steps,False
11,Not sure but maybe around 4-5 kms a day,NaN,NaN,NaN,unknown,False
13,2k,NaN,NaN,NaN,probable_steps,False
20,7-9k,NaN,NaN,NaN,probable_steps,False
24,5-6K,NaN,NaN,NaN,probable_steps,False
40,10000 to 15000,NaN,NaN,NaN,probable_steps,False
42,20 maybe,NaN,NaN,NaN,probable_minutes,False
56,10k+,NaN,NaN,NaN,probable_steps,True



========== DIETARY ENCODING ==========
Dietary binary features: 7
All dietary features contain only 0 and 1.

========== ORIGINAL DATA PRESERVATION ==========
Original columns unchanged: True

========== FINAL AUDIT SUMMARY ==========


,Check,Result
0,Dataset rows,100
1,Dataset columns,42
2,Duplicate respondent IDs,0
3,Missing respondent IDs,0
4,Missing academic screen values,3
5,Missing sitting duration values,6
6,Dietary binary features,7
7,Corrected step-count records,1
8,Lower-bound activity responses,2
9,Activity responses flagged for review,16



Files saved successfully!
Final dataset shape: (100, 42)
Reloaded shape: (100, 32)
Shapes match: False
